# 新Patient预测 - KAN & DeepMLP

本notebook对新patient数据使用训练好的KAN和DeepMLP模型进行预测，并保存结果到各自的模型目录。

**模型**:
- KAN: `/training_runs/kan_bg_excl_20251103_162620/results/`
- DeepMLP: `/training_runs/deep_mlp_bg_excl_20251103_162620/results/`

**新数据**:
- 特征: `stacked_zscore_float32.nii.gz` (已标准化)
- 标签: `resampled_synthseg_t1_mp2rage_alex_labels.nii.gz` (FreeSurfer标签)

**处理流程**:
1. 加载新patient数据（特征+标签）
2. 标签映射（FreeSurfer -> 0-51连续索引）
3. 排除背景（label=0）
4. 排除feature 14
5. 对每个模型预测并保存3D softmax volume

## Cell 1: 导入库和全局配置

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import nibabel as nib
from pathlib import Path
from datetime import datetime
from tqdm import tqdm
import json
import sys

# 设置设备
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️ 使用设备: {device}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   显存: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

# FreeSurfer标准标签（52个：0-51）
STANDARD_LABELS = [
    0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 17,
    29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43,
    44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58,
    59, 60, 61, 62, 103
]

# 创建标签映射
def create_label_mapping():
    """创建FreeSurfer标签到连续索引的映射"""
    forward_mapping = {original: continuous for continuous, original in enumerate(STANDARD_LABELS)}
    reverse_mapping = {continuous: original for continuous, original in enumerate(STANDARD_LABELS)}
    return forward_mapping, reverse_mapping

forward_mapping, reverse_mapping = create_label_mapping()
print(f"\n📋 标签映射: {len(STANDARD_LABELS)}个类别 (0-51)")
print(f"   示例: FreeSurfer {STANDARD_LABELS[:5]} -> 连续索引 [0,1,2,3,4]")

print("\n✅ 导入完成！")

## Cell 2: 配置路径

In [ ]:
# =============================================================================
# 配置区域 - 根据实际情况修改
# =============================================================================

# 服务器路径配置
SERVER_BASE = '/home/jovyan/gpu_space/workspace_jiayi/KAN-git/KAN-Brain-Single-Voxel-Segmentaion/D_proj_analysis/banlanced_sample'

# 模型路径配置
MODELS = {
    'KAN': {
        'model_file': f'{SERVER_BASE}/refactored_training/training_runs/kan_bg_excl_20251103_162620/results/kan_bg_excl_20251103_164641.pth',
        'output_dir': f'{SERVER_BASE}/refactored_training/training_runs/kan_bg_excl_20251103_162620/results',
        'model_type': 'kan'
    },
    'DeepMLP': {
        'model_file': f'{SERVER_BASE}/refactored_training/training_runs/deep_mlp_bg_excl_20251103_162620/results/deep_mlp_bg_excl_20251103_200903.pth',
        'output_dir': f'{SERVER_BASE}/refactored_training/training_runs/deep_mlp_bg_excl_20251103_162620/results',
        'model_type': 'deep_mlp'
    }
}

# 新patient数据路径
NEW_PATIENT_DATA = {
    'features': '/home/jovyan/gpu_space/workspace_jiayi/new_datasets/NEW_DATASET_ANALYSIS/FOR_016_20250204_reproducibility/evaluated/stacked_zscore_float32.nii.gz',
    'labels': '/home/jovyan/gpu_space/workspace_jiayi/new_datasets/NEW_DATASET_ANALYSIS/FOR_016_20250204_reproducibility/evaluated/resampled_synthseg_t1_mp2rage_alex_labels.nii.gz',
    'subject_id': 'FOR_016_20250204_reproducibility'
}

# 预测参数
PREDICTION_CONFIG = {
    'batch_size': 8192,
    'exclude_features': [14],  # 排除feature 14
    'include_background': False,  # 排除背景
    'device': device
}

# 验证路径
print("📁 路径配置验证:")
print("\n模型文件:")
for model_name, config in MODELS.items():
    model_path = Path(config['model_file'])
    if model_path.exists():
        size_mb = model_path.stat().st_size / (1024**2)
        print(f"  ✅ {model_name}: {model_path.name} ({size_mb:.1f} MB)")
    else:
        print(f"  ❌ {model_name}: 文件未找到!")
        print(f"     {config['model_file']}")

print("\n新patient数据:")
for data_type, path in NEW_PATIENT_DATA.items():
    if data_type == 'subject_id':
        continue
    data_path = Path(path)
    if data_path.exists():
        size_mb = data_path.stat().st_size / (1024**2)
        print(f"  ✅ {data_type}: {data_path.name} ({size_mb:.1f} MB)")
    else:
        print(f"  ❌ {data_type}: 文件未找到!")
        print(f"     {path}")

print(f"\n⚙️ 预测配置:")
print(f"  批大小: {PREDICTION_CONFIG['batch_size']}")
print(f"  排除特征: {PREDICTION_CONFIG['exclude_features']}")
print(f"  排除背景: {PREDICTION_CONFIG['include_background'] == False}")
print(f"  设备: {PREDICTION_CONFIG['device']}")

print("\n✅ 配置完成！")

## Cell 3: 定义数据加载函数

In [ ]:
def load_new_patient_data(feature_path, label_path, subject_id,
                          forward_mapping, include_background=False,
                          exclude_features=None):
    """
    加载新patient数据并进行预处理
    
    Parameters:
    -----------
    feature_path : str
        4D特征文件路径 (X,Y,Z,n_features)
    label_path : str
        3D标签文件路径 (X,Y,Z)
    subject_id : str
        受试者ID
    forward_mapping : dict
        FreeSurfer标签到连续索引的映射
    include_background : bool
        是否包含背景体素
    exclude_features : list
        要排除的特征索引
    
    Returns:
    --------
    dict : 包含处理后的数据和元信息
    """
    print(f"\n📂 加载新patient数据: {subject_id}")
    
    # 1. 加载特征数据
    print(f"  加载特征: {Path(feature_path).name}")
    feature_img = nib.load(feature_path)
    features_4d = feature_img.get_fdata().astype(np.float32)
    print(f"    形状: {features_4d.shape}")
    print(f"    范围: [{features_4d.min():.3f}, {features_4d.max():.3f}]")
    
    # 2. 加载标签数据
    print(f"  加载标签: {Path(label_path).name}")
    label_img = nib.load(label_path)
    labels_3d = label_img.get_fdata().astype(np.int32)
    print(f"    形状: {labels_3d.shape}")
    
    original_shape_3d = labels_3d.shape
    
    # 验证形状匹配
    assert features_4d.shape[:3] == labels_3d.shape, "特征和标签的3D形状不匹配！"
    
    # 3. 展平数据
    n_voxels = np.prod(labels_3d.shape)
    n_modalities = features_4d.shape[3]
    
    features_flat = features_4d.reshape(n_voxels, n_modalities)
    labels_flat = labels_3d.flatten()
    
    print(f"  展平: {n_voxels:,} 体素 × {n_modalities} 特征")
    
    # 4. 标签映射
    print(f"  标签映射...")
    unique_orig = np.unique(labels_flat)
    print(f"    原始标签: {len(unique_orig)} 个唯一值")
    print(f"    {unique_orig[:10]}...")
    
    labels_mapped = np.zeros_like(labels_flat)
    unmapped_labels = []
    
    for orig_label in unique_orig:
        if orig_label in forward_mapping:
            mask = labels_flat == orig_label
            labels_mapped[mask] = forward_mapping[orig_label]
            count = np.sum(mask)
            if orig_label != 0:  # 不显示背景
                print(f"      {orig_label:3d} -> {forward_mapping[orig_label]:2d} ({count:,} 体素)")
        else:
            unmapped_labels.append(orig_label)
    
    if unmapped_labels:
        print(f"    ⚠️ 未映射的标签: {unmapped_labels}")
    
    labels_flat = labels_mapped
    
    # 5. 处理背景
    if not include_background:
        # 排除背景体素（label=0）
        spatial_mask_flat = labels_flat != 0
        spatial_mask_3d = spatial_mask_flat.reshape(original_shape_3d)
        flat_indices = np.where(spatial_mask_flat)[0]
        
        features_flat = features_flat[spatial_mask_flat]
        labels_flat = labels_flat[spatial_mask_flat]
        
        print(f"  排除背景: {len(features_flat):,} / {n_voxels:,} 体素 "
              f"({len(features_flat)/n_voxels*100:.2f}%)")
    else:
        spatial_mask_3d = np.ones(original_shape_3d, dtype=bool)
        flat_indices = np.arange(n_voxels)
        print(f"  包含背景: {len(features_flat):,} 体素")
    
    # 6. 排除指定特征
    if exclude_features:
        keep_mask = np.ones(features_flat.shape[1], dtype=bool)
        keep_mask[exclude_features] = False
        features_flat = features_flat[:, keep_mask]
        print(f"  排除特征{exclude_features}: 保留 {features_flat.shape[1]} 个特征")
    
    # 7. 统计信息
    print(f"\n  📊 数据统计:")
    print(f"    最终形状: {features_flat.shape}")
    print(f"    特征范围: [{features_flat.min():.3f}, {features_flat.max():.3f}]")
    print(f"    特征均值: {features_flat.mean():.3f}")
    print(f"    特征标准差: {features_flat.std():.3f}")
    print(f"    标签范围: [{labels_flat.min()}, {labels_flat.max()}]")
    print(f"    唯一标签数: {len(np.unique(labels_flat))}")
    
    return {
        'features': features_flat,
        'labels': labels_flat,
        'subject_id': subject_id,
        'n_voxels': len(features_flat),
        'original_shape_3d': original_shape_3d,
        'spatial_mask_3d': spatial_mask_3d,
        'flat_indices': flat_indices,
        'affine': feature_img.affine,
        'header': feature_img.header
    }

print("✅ 数据加载函数定义完成！")

## Cell 4: 定义模型加载函数

In [ ]:
# 添加refactored_training目录到路径
refactored_dir = Path('/home/jovyan/gpu_space/workspace_jiayi/KAN-git/KAN-Brain-Single-Voxel-Segmentaion/D_proj_analysis/banlanced_sample/refactored_training')
if str(refactored_dir) not in sys.path:
    sys.path.insert(0, str(refactored_dir))

# 导入模型
from models import get_model

def load_trained_model(model_path, model_type, device):
    """
    加载训练好的模型
    
    Parameters:
    -----------
    model_path : str
        模型.pth文件路径
    model_type : str
        模型类型 ('kan' 或 'deep_mlp')
    device : torch.device
        设备
    
    Returns:
    --------
    model : nn.Module
        加载的模型
    checkpoint : dict
        checkpoint信息
    """
    print(f"\n🔧 加载模型: {model_type}")
    print(f"  文件: {Path(model_path).name}")
    
    # 加载checkpoint
    checkpoint = torch.load(model_path, map_location=device)
    
    # 提取配置
    config = checkpoint.get('config', {})
    input_dim = config.get('input_dim', 41)  # 排除feature 14后
    num_classes = config.get('num_classes', 52)
    
    print(f"  配置: input_dim={input_dim}, num_classes={num_classes}")
    
    # 创建模型实例（get_model返回(model, config)元组）
    if model_type == 'kan':
        model, _ = get_model(
            'kan',
            input_dim=input_dim,
            num_classes=num_classes,
            grid_size=config.get('grid_size', 8)
        )
    elif model_type == 'deep_mlp':
        model, _ = get_model(
            'deep_mlp',
            input_dim=input_dim,
            num_classes=num_classes
        )
    else:
        raise ValueError(f"不支持的模型类型: {model_type}")
    
    # 加载权重
    model.load_state_dict(checkpoint['model_state_dict'])
    model = model.to(device)
    model.eval()
    
    # 统计参数
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    print(f"  参数量: {total_params:,} (可训练: {trainable_params:,})")
    print(f"  Epoch: {checkpoint.get('epoch', 'N/A')}")
    
    history = checkpoint.get('history', {})
    if history:
        test_f1 = history.get('test_f1', [])
        if test_f1:
            print(f"  Test F1: {test_f1[-1]:.4f} (最终)")
    
    print(f"  ✅ 模型加载完成！")
    
    return model, checkpoint

print("✅ 模型加载函数定义完成！")

## Cell 5: 定义预测和保存函数

In [ ]:
def predict_patient(model, data_info, device, batch_size=8192):
    """
    对patient数据进行预测
    
    Parameters:
    -----------
    model : nn.Module
        训练好的模型
    data_info : dict
        数据信息（来自load_new_patient_data）
    device : torch.device
        设备
    batch_size : int
        批大小
    
    Returns:
    --------
    predictions : np.ndarray
        Softmax概率 (n_voxels, n_classes)
    """
    print(f"\n🔮 开始预测...")
    
    features = data_info['features']
    n_samples = len(features)
    
    model.eval()
    all_predictions = []
    
    with torch.no_grad():
        for start_idx in tqdm(range(0, n_samples, batch_size), desc="预测进度"):
            end_idx = min(start_idx + batch_size, n_samples)
            
            # 准备batch
            batch_X = torch.FloatTensor(features[start_idx:end_idx]).to(device)
            
            # 前向传播
            logits = model(batch_X)
            
            # 转换为softmax概率
            probabilities = F.softmax(logits, dim=1)
            
            all_predictions.append(probabilities.cpu().numpy())
            
            # 清理内存
            del batch_X, logits, probabilities
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
    
    # 合并所有预测
    predictions = np.vstack(all_predictions)
    
    print(f"  ✅ 预测完成！")
    print(f"    形状: {predictions.shape}")
    print(f"    概率范围: [{predictions.min():.6f}, {predictions.max():.6f}]")
    print(f"    概率和: [{np.sum(predictions, axis=1).min():.6f}, {np.sum(predictions, axis=1).max():.6f}]")
    
    return predictions


def predictions_to_3d_volume(predictions, data_info, include_background=False):
    """
    将预测结果还原为3D softmax volume
    
    Parameters:
    -----------
    predictions : np.ndarray
        模型预测的softmax概率 (n_valid_voxels, n_classes)
    data_info : dict
        数据信息
    include_background : bool
        训练时是否包含了背景
    
    Returns:
    --------
    volume_3d : np.ndarray
        3D softmax volume (X, Y, Z, n_classes)
    """
    print(f"\n🔄 还原3D volume...")
    
    original_shape_3d = data_info['original_shape_3d']
    spatial_mask_3d = data_info['spatial_mask_3d']
    flat_indices = data_info['flat_indices']
    
    n_classes = predictions.shape[1]
    
    # 创建完整的3D softmax volume
    volume_3d = np.zeros((*original_shape_3d, n_classes), dtype=np.float32)
    
    if include_background:
        # 如果训练时包含了背景，直接映射
        volume_flat = volume_3d.reshape(-1, n_classes)
        volume_flat[flat_indices] = predictions
    else:
        # 如果训练时排除了背景，需要特殊处理
        volume_flat = volume_3d.reshape(-1, n_classes)
        volume_flat[flat_indices] = predictions
        
        # 背景区域：class 0概率=1，其他=0
        background_indices = np.where(~spatial_mask_3d.flatten())[0]
        volume_flat[background_indices, 0] = 1.0
    
    print(f"  ✅ 3D volume形状: {volume_3d.shape}")
    print(f"    概率和范围: [{np.sum(volume_3d, axis=-1).min():.6f}, {np.sum(volume_3d, axis=-1).max():.6f}]")
    
    return volume_3d


def save_prediction_results(volume_3d, data_info, output_dir, model_name,
                           include_background=False):
    """
    保存预测结果（NIfTI + JSON）
    
    Parameters:
    -----------
    volume_3d : np.ndarray
        3D softmax volume
    data_info : dict
        数据信息
    output_dir : str
        输出目录
    model_name : str
        模型名称
    include_background : bool
        是否包含背景
    
    Returns:
    --------
    nifti_path : Path
        NIfTI文件路径
    json_path : Path
        JSON信息文件路径
    """
    print(f"\n💾 保存预测结果...")
    
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    subject_id = data_info['subject_id']
    bg_str = 'incl' if include_background else 'excl'
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    
    # 使用原始的仿射矩阵和头信息
    affine = data_info['affine']
    header = data_info['header'].copy()
    
    # 更新头信息（4D数据）
    header.set_data_shape(volume_3d.shape)
    header.set_data_dtype(np.float32)
    
    # 创建NIfTI图像
    nifti_img = nib.Nifti1Image(volume_3d, affine, header)
    
    # 保存NIfTI文件
    nifti_path = output_dir / f"prediction_3d_{model_name}_{subject_id}_bg_{bg_str}_{timestamp}.nii.gz"
    nib.save(nifti_img, nifti_path)
    
    file_size_mb = nifti_path.stat().st_size / (1024**2)
    print(f"  ✅ NIfTI已保存: {nifti_path.name}")
    print(f"    大小: {file_size_mb:.2f} MB")
    
    # 保存元信息JSON
    info_dict = {
        'model_name': model_name,
        'subject_id': subject_id,
        'timestamp': timestamp,
        'prediction_shape': volume_3d.shape,
        'original_shape_3d': data_info['original_shape_3d'],
        'include_background': include_background,
        'n_valid_voxels': data_info['n_voxels'],
        'n_classes': volume_3d.shape[-1],
        'prediction_stats': {
            'min_prob': float(volume_3d.min()),
            'max_prob': float(volume_3d.max()),
            'mean_prob': float(volume_3d.mean())
        },
        'class_labels': STANDARD_LABELS
    }
    
    json_path = output_dir / f"prediction_info_{model_name}_{subject_id}_bg_{bg_str}_{timestamp}.json"
    with open(json_path, 'w') as f:
        json.dump(info_dict, f, indent=2)
    
    print(f"  ✅ JSON已保存: {json_path.name}")
    print(f"\n  📁 保存位置: {output_dir}")
    
    return nifti_path, json_path

print("✅ 预测和保存函数定义完成！")

## Cell 6: 加载新patient数据

In [ ]:
# 加载新patient数据
patient_data = load_new_patient_data(
    feature_path=NEW_PATIENT_DATA['features'],
    label_path=NEW_PATIENT_DATA['labels'],
    subject_id=NEW_PATIENT_DATA['subject_id'],
    forward_mapping=forward_mapping,
    include_background=PREDICTION_CONFIG['include_background'],
    exclude_features=PREDICTION_CONFIG['exclude_features']
)

print("\n" + "="*80)
print("✅ 新patient数据加载完成！")
print("="*80)

## Cell 7: 预测 - KAN模型

In [ ]:
print("\n" + "="*80)
print("🚀 开始预测 - KAN模型")
print("="*80)

# 1. 加载KAN模型
kan_model, kan_checkpoint = load_trained_model(
    model_path=MODELS['KAN']['model_file'],
    model_type=MODELS['KAN']['model_type'],
    device=device
)

# 2. 执行预测
kan_predictions = predict_patient(
    model=kan_model,
    data_info=patient_data,
    device=device,
    batch_size=PREDICTION_CONFIG['batch_size']
)

# 3. 还原为3D volume
kan_volume_3d = predictions_to_3d_volume(
    predictions=kan_predictions,
    data_info=patient_data,
    include_background=PREDICTION_CONFIG['include_background']
)

# 4. 保存结果
kan_nifti_path, kan_json_path = save_prediction_results(
    volume_3d=kan_volume_3d,
    data_info=patient_data,
    output_dir=MODELS['KAN']['output_dir'],
    model_name='kan',
    include_background=PREDICTION_CONFIG['include_background']
)

# 5. 清理内存 (保留 kan_predictions 供下一个 cell 使用)
del kan_model, kan_volume_3d
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("\n" + "="*80)
print("✅ KAN模型预测完成！")
print(f"💡 kan_predictions 已保留在内存中供 error map 可视化使用")
print("="*80)

In [ ]:
# ==============================================================================
# ERROR MAP VISUALIZATION - KAN MODEL (USING IN-MEMORY DATA)
# This cell must run immediately after Cell 7 (KAN prediction)
# ==============================================================================

import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

print("="*80)
print("ERROR MAP VISUALIZATION - KAN MODEL (IN-MEMORY)")
print("="*80)

# ==============================================================================
# STEP 1: Get Predicted Labels from In-Memory Softmax
# ==============================================================================

print("\n📊 Processing in-memory predictions...")

# Get predicted labels (argmax of softmax probabilities)
# kan_predictions shape: (n_valid_voxels, 52)
pred_labels_flat = np.argmax(kan_predictions, axis=1).astype(np.int32)
print(f"  Predicted labels (flat): {pred_labels_flat.shape}")
print(f"  Predicted label range: [{pred_labels_flat.min()}, {pred_labels_flat.max()}]")

# Get ground truth labels (already mapped to 0-51)
# patient_data['labels'] shape: (n_valid_voxels,)
gt_labels_flat = patient_data['labels'].astype(np.int32)
print(f"  Ground truth labels (flat): {gt_labels_flat.shape}")
print(f"  Ground truth label range: [{gt_labels_flat.min()}, {gt_labels_flat.max()}]")

# Verify they have the same length
assert len(pred_labels_flat) == len(gt_labels_flat), \
    f"Length mismatch: pred {len(pred_labels_flat)} vs gt {len(gt_labels_flat)}"

# ==============================================================================
# STEP 2: Calculate Accuracy on Valid Voxels (Excluding Background)
# ==============================================================================

print("\n🎨 Creating error map...")

# Non-background mask (these are already non-background voxels)
non_bg_mask = gt_labels_flat > 0

# Calculate accuracy
correct_mask = (pred_labels_flat == gt_labels_flat) & non_bg_mask
incorrect_mask = (pred_labels_flat != gt_labels_flat) & non_bg_mask

n_correct = correct_mask.sum()
n_total = non_bg_mask.sum()
gross_accuracy = n_correct / n_total if n_total > 0 else 0.0

print(f"  Total non-background voxels: {n_total:,}")
print(f"  Correct predictions: {n_correct:,}")
print(f"  Incorrect predictions: {incorrect_mask.sum():,}")
print(f"  ✅ Gross Accuracy: {gross_accuracy:.4f} ({gross_accuracy*100:.2f}%)")

# ==============================================================================
# STEP 3: Reconstruct 3D Error Map
# ==============================================================================

print("\n🔄 Reconstructing 3D error map...")

# Get spatial info
original_shape_3d = patient_data['original_shape_3d']
spatial_mask_3d = patient_data['spatial_mask_3d']
flat_indices = patient_data['flat_indices']

print(f"  Original 3D shape: {original_shape_3d}")
print(f"  Number of valid voxels: {len(flat_indices):,}")

# Create error map
# 0 = background (gray)
# 1 = correct prediction (green)
# 2 = incorrect prediction (red)
error_map_3d = np.zeros(original_shape_3d, dtype=np.uint8)

# Flatten the error map
error_map_flat = error_map_3d.flatten()

# Mark correct predictions (green) - FIXED: removed double indexing
correct_indices = flat_indices[correct_mask]
error_map_flat[correct_indices] = 1

# Mark incorrect predictions (red) - FIXED: removed double indexing
incorrect_indices = flat_indices[incorrect_mask]
error_map_flat[incorrect_indices] = 2

# Reshape back to 3D
error_map_3d = error_map_flat.reshape(original_shape_3d)

print(f"  Error map created: {error_map_3d.shape}")
print(f"  Background voxels: {(error_map_3d == 0).sum():,}")
print(f"  Correct voxels: {(error_map_3d == 1).sum():,}")
print(f"  Incorrect voxels: {(error_map_3d == 2).sum():,}")

# Sanity check
assert (error_map_3d == 1).sum() == n_correct, "Correct voxel count mismatch!"
assert (error_map_3d == 2).sum() == incorrect_mask.sum(), "Incorrect voxel count mismatch!"

# ==============================================================================
# STEP 4: Calculate Per-Class Metrics
# ==============================================================================

print("\n📊 Per-class prediction summary (top 10 by frequency):")

unique_gt, counts_gt = np.unique(gt_labels_flat[non_bg_mask], return_counts=True)
unique_pred, counts_pred = np.unique(pred_labels_flat[non_bg_mask], return_counts=True)

# Calculate per-class accuracy
class_accuracies = {}
for class_id in unique_gt:
    class_mask = gt_labels_flat == class_id
    if class_mask.sum() > 0:
        class_correct = (pred_labels_flat[class_mask] == class_id).sum()
        class_acc = class_correct / class_mask.sum()
        class_accuracies[class_id] = (class_acc, class_mask.sum(), class_correct)

# Sort by frequency and show top 10
sorted_classes = sorted(class_accuracies.items(), key=lambda x: x[1][1], reverse=True)[:10]
for class_id, (acc, total, correct) in sorted_classes:
    print(f"  Class {class_id:2d}: {acc:.3f} ({correct:6,}/{total:6,}) - FreeSurfer label {STANDARD_LABELS[class_id]}")

# ==============================================================================
# STEP 5: Select Slices to Visualize
# ==============================================================================

N_SLICES = 5
X, Y, Z = original_shape_3d

print(f"\n🖼️  Selecting {N_SLICES} slices per view...")

# Select evenly spaced slices
z_slices = np.linspace(Z//4, 3*Z//4, N_SLICES, dtype=int)
y_slices = np.linspace(Y//4, 3*Y//4, N_SLICES, dtype=int)
x_slices = np.linspace(X//4, 3*X//4, N_SLICES, dtype=int)

print(f"  Axial (Z): {z_slices.tolist()}")
print(f"  Coronal (Y): {y_slices.tolist()}")
print(f"  Sagittal (X): {x_slices.tolist()}")

# ==============================================================================
# STEP 6: Visualize Error Maps
# ==============================================================================

print("\n📊 Generating visualization...")

# Reconstruct GT labels in 3D for slice accuracy calculation
gt_labels_3d = np.zeros(original_shape_3d, dtype=np.int32)
gt_labels_3d_flat = gt_labels_3d.flatten()
gt_labels_3d_flat[flat_indices] = gt_labels_flat
gt_labels_3d = gt_labels_3d_flat.reshape(original_shape_3d)

# Define colormap
colors = ['#2C2C2C', '#00FF00', '#FF0000']  # gray, green, red
cmap = ListedColormap(colors)

# Create figure
fig, axes = plt.subplots(3, N_SLICES, figsize=(20, 12))
fig.suptitle(f'KAN Model - Error Map Visualization (In-Memory Data)\\n'
             f'Gross Accuracy: {gross_accuracy:.4f} ({gross_accuracy*100:.2f}%)\\n'
             f'Green = Correct | Red = Incorrect | Gray = Background',
             fontsize=16, fontweight='bold', y=0.98)

# Plot axial slices
for i, z_idx in enumerate(z_slices):
    ax = axes[0, i]
    slice_data = error_map_3d[:, :, z_idx].T
    
    # Calculate slice accuracy
    slice_mask = gt_labels_3d[:, :, z_idx] > 0
    if slice_mask.sum() > 0:
        slice_correct = (error_map_3d[:, :, z_idx][slice_mask] == 1).sum()
        slice_acc = slice_correct / slice_mask.sum()
    else:
        slice_acc = 0.0
    
    ax.imshow(slice_data, cmap=cmap, vmin=0, vmax=2, interpolation='nearest')
    ax.set_title(f'Axial Z={z_idx}\\nAcc: {slice_acc:.3f}', fontsize=10)
    ax.axis('off')

# Plot coronal slices
for i, y_idx in enumerate(y_slices):
    ax = axes[1, i]
    slice_data = error_map_3d[:, y_idx, :].T
    
    slice_mask = gt_labels_3d[:, y_idx, :] > 0
    if slice_mask.sum() > 0:
        slice_correct = (error_map_3d[:, y_idx, :][slice_mask] == 1).sum()
        slice_acc = slice_correct / slice_mask.sum()
    else:
        slice_acc = 0.0
    
    ax.imshow(slice_data, cmap=cmap, vmin=0, vmax=2, interpolation='nearest')
    ax.set_title(f'Coronal Y={y_idx}\\nAcc: {slice_acc:.3f}', fontsize=10)
    ax.axis('off')

# Plot sagittal slices
for i, x_idx in enumerate(x_slices):
    ax = axes[2, i]
    slice_data = error_map_3d[x_idx, :, :].T
    
    slice_mask = gt_labels_3d[x_idx, :, :] > 0
    if slice_mask.sum() > 0:
        slice_correct = (error_map_3d[x_idx, :, :][slice_mask] == 1).sum()
        slice_acc = slice_correct / slice_mask.sum()
    else:
        slice_acc = 0.0
    
    ax.imshow(slice_data, cmap=cmap, vmin=0, vmax=2, interpolation='nearest')
    ax.set_title(f'Sagittal X={x_idx}\\nAcc: {slice_acc:.3f}', fontsize=10)
    ax.axis('off')

plt.tight_layout()
plt.show()

print("\n" + "="*80)
print("✅ Visualization complete!")
print("="*80)
print(f"\n📊 Summary:")
print(f"  Model: KAN")
print(f"  Volume shape: {original_shape_3d}")
print(f"  Total non-background voxels: {n_total:,}")
print(f"  Correct predictions: {n_correct:,} ({n_correct/n_total*100:.2f}%)")
print(f"  Incorrect predictions: {incorrect_mask.sum():,} ({incorrect_mask.sum()/n_total*100:.2f}%)")
print(f"  Overall Accuracy: {gross_accuracy:.4f}")

if gross_accuracy < 0.3:
    print("\n" + "="*80)
    print("⚠️  WARNING: Very low accuracy detected!")
    print("="*80)
    print("Possible causes:")
    print("  1. Data standardization mismatch (most likely)")
    print("  2. Label mapping inconsistency")
    print("  3. Different data distribution than training")
    print("\nPlease check data standardization in the next cell.")
    print("="*80)

print("="*80)

## Cell 7c: 标签映射验证 (Label Mapping Verification)

**重要提示**: 运行此cell验证GT labels是否正确映射到0-51连续索引。

In [ ]:
# ==============================================================================
# LABEL MAPPING VERIFICATION
# This cell verifies that GT labels are correctly mapped to 0-51
# ==============================================================================

print("="*80)
print("LABEL MAPPING VERIFICATION")
print("="*80)

# ==============================================================================
# Check 1: Verify patient_data labels are mapped
# ==============================================================================

print("\n📊 Checking patient_data['labels'] (should be 0-51):")

gt_labels_flat = patient_data['labels'].astype(np.int32)
unique_gt = np.unique(gt_labels_flat)

print(f"  Shape: {gt_labels_flat.shape}")
print(f"  Unique values count: {len(unique_gt)}")
print(f"  Range: [{gt_labels_flat.min()}, {gt_labels_flat.max()}]")
print(f"  All unique values: {sorted(unique_gt.tolist())}")

# Check if labels are in expected range (0-51)
expected_range = all(0 <= label <= 51 for label in unique_gt)

if expected_range:
    print(f"\n  ✅ Labels are in expected range [0, 51]")
else:
    print(f"\n  ❌ WARNING: Labels are NOT in expected range!")
    print(f"     Some labels are > 51, indicating they might be unmapped FreeSurfer labels!")

# ==============================================================================
# Check 2: Compare with prediction labels
# ==============================================================================

print("\n📊 Checking KAN prediction labels (should also be 0-51):")

pred_labels_flat = np.argmax(kan_predictions, axis=1).astype(np.int32)
unique_pred = np.unique(pred_labels_flat)

print(f"  Shape: {pred_labels_flat.shape}")
print(f"  Unique values count: {len(unique_pred)}")
print(f"  Range: [{pred_labels_flat.min()}, {pred_labels_flat.max()}]")
print(f"  All unique values: {sorted(unique_pred.tolist())}")

# ==============================================================================
# Check 3: Verify mapping consistency
# ==============================================================================

print("\n" + "="*80)
print("MAPPING CONSISTENCY CHECK")
print("="*80)

# Check if GT and pred have overlapping label sets
gt_set = set(unique_gt)
pred_set = set(unique_pred)

overlap = gt_set & pred_set
only_gt = gt_set - pred_set
only_pred = pred_set - gt_set

print(f"\n  Labels in both GT and Pred: {len(overlap)}")
print(f"  Labels only in GT: {len(only_gt)}")
if only_gt:
    print(f"    {sorted(only_gt)[:10]}{'...' if len(only_gt) > 10 else ''}")
print(f"  Labels only in Pred: {len(only_pred)}")
if only_pred:
    print(f"    {sorted(only_pred)[:10]}{'...' if len(only_pred) > 10 else ''}")

# ==============================================================================
# Check 4: Re-load original labels and verify mapping
# ==============================================================================

print("\n" + "="*80)
print("ORIGINAL LABELS CHECK")
print("="*80)

print("\n📂 Re-loading original label file to verify mapping...")

import nibabel as nib
label_img = nib.load(NEW_PATIENT_DATA['labels'])
original_labels = label_img.get_fdata().astype(np.int32).flatten()

# Get non-background labels only
original_nonbg = original_labels[original_labels > 0]
unique_original = np.unique(original_nonbg)

print(f"\n  Original labels (before mapping):")
print(f"    Unique values: {len(unique_original)}")
print(f"    Range: [{unique_original.min()}, {unique_original.max()}]")
print(f"    Values: {sorted(unique_original.tolist())}")

print(f"\n  Mapped labels (from patient_data):")
gt_nonbg = gt_labels_flat[gt_labels_flat > 0]
unique_mapped = np.unique(gt_nonbg)
print(f"    Unique values: {len(unique_mapped)}")
print(f"    Range: [{unique_mapped.min()}, {unique_mapped.max()}]")
print(f"    Values: {sorted(unique_mapped.tolist())}")

# ==============================================================================
# Check 5: Verify forward_mapping
# ==============================================================================

print("\n" + "="*80)
print("FORWARD MAPPING VERIFICATION")
print("="*80)

print("\n  Verifying each original label maps correctly:")
all_mapped_correctly = True

for orig in sorted(unique_original)[:10]:  # Show first 10
    if orig in forward_mapping:
        mapped = forward_mapping[orig]
        # Count occurrences
        orig_count = (original_nonbg == orig).sum()
        mapped_count = (gt_nonbg == mapped).sum()
        
        match_status = "✅" if orig_count == mapped_count else "❌"
        print(f"    {match_status} {orig:3d} -> {mapped:2d}: {orig_count:7,} voxels (original) vs {mapped_count:7,} voxels (mapped)")
        
        if orig_count != mapped_count:
            all_mapped_correctly = False
    else:
        print(f"    ❌ {orig:3d} -> NOT IN MAPPING!")
        all_mapped_correctly = False

if len(unique_original) > 10:
    print(f"    ... ({len(unique_original) - 10} more labels)")

# ==============================================================================
# Check 6: Final Diagnosis
# ==============================================================================

print("\n" + "="*80)
print("DIAGNOSIS")
print("="*80)

if expected_range and all_mapped_correctly:
    print("\n✅ LABEL MAPPING IS CORRECT")
    print("  - GT labels are properly mapped to 0-51")
    print("  - Prediction labels are in range 0-51")
    print("  - All original labels are correctly mapped")
    print("\n  Low accuracy is NOT due to label mapping issue.")
    print("  Please run Cell 7d (Data Standardization Diagnostic) to check other causes.")
else:
    print("\n❌ LABEL MAPPING HAS PROBLEMS!")
    
    if not expected_range:
        print("\n  Problem 1: GT labels are not in expected range [0-51]")
        print("  This means labels were NOT correctly mapped in load_new_patient_data()")
        print("\n  SOLUTION: Check the forward_mapping and ensure all FreeSurfer labels are mapped.")
    
    if not all_mapped_correctly:
        print("\n  Problem 2: Voxel counts don't match between original and mapped labels")
        print("  This indicates mapping errors in the load_new_patient_data() function.")
    
    print("\n  This is DEFINITELY causing the low accuracy!")

print("\n" + "="*80)

## Cell 7d: 数据标准化诊断 (Data Standardization Diagnostic)

**重要提示**: 如果标签映射正确但accuracy仍然很低，运行此cell检查数据标准化问题。

In [ ]:
# ==============================================================================
# DATA STANDARDIZATION DIAGNOSTIC
# Run this cell if label mapping is correct but accuracy is still low
# ==============================================================================

print("="*80)
print("DATA STANDARDIZATION DIAGNOSTIC")
print("="*80)

# ==============================================================================
# Check 1: Current Data Statistics
# ==============================================================================

print("\n📊 Current data statistics (after loading):")
print(f"\n  Overall (all voxels, all modalities):")
print(f"    Mean: {patient_data['features'].mean():.6f}")
print(f"    Std:  {patient_data['features'].std():.6f}")
print(f"    Min:  {patient_data['features'].min():.6f}")
print(f"    Max:  {patient_data['features'].max():.6f}")

print(f"\n  Per-modality statistics:")
n_modalities = patient_data['features'].shape[1]
for i in range(min(n_modalities, 10)):  # Show first 10 modalities
    feat = patient_data['features'][:, i]
    print(f"    Modality {i:2d}: mean={feat.mean():7.4f}, std={feat.std():7.4f}, "
          f"min={feat.min():7.2f}, max={feat.max():7.2f}")
if n_modalities > 10:
    print(f"    ... ({n_modalities - 10} more modalities)")

# ==============================================================================
# Check 2: Expected Values for Patient-Wise Standardization
# ==============================================================================

print("\n" + "="*80)
print("EXPECTED VALUES (from training)")
print("="*80)

print("\n✅ For patient-wise z-score standardization:")
print("    Overall mean ≈ 0.0")
print("    Overall std  ≈ 1.0")
print("\n  Formula: (features - patient_mean) / patient_std")

# ==============================================================================
# Check 3: Training Checkpoint Scaler Info
# ==============================================================================

print("\n" + "="*80)
print("TRAINING CHECKPOINT INFORMATION")
print("="*80)

if 'scalers_info' in kan_checkpoint:
    print("\n📦 Scaler info found in checkpoint:")
    scalers = kan_checkpoint['scalers_info']
    print(f"  Type: {type(scalers)}")
    
    if isinstance(scalers, dict):
        for key, value in list(scalers.items())[:5]:
            print(f"  {key}: {value}")
    else:
        print(f"  Content: {scalers}")
else:
    print("\n⚠️  No 'scalers_info' found in checkpoint")
    print("  Available keys:", list(kan_checkpoint.keys()))

# ==============================================================================
# Check 4: Diagnosis and Recommendation
# ==============================================================================

print("\n" + "="*80)
print("DIAGNOSIS")
print("="*80)

current_mean = patient_data['features'].mean()
current_std = patient_data['features'].std()

# Check if standardization matches expected values
mean_ok = abs(current_mean) < 0.5  # Should be close to 0
std_ok = abs(current_std - 1.0) < 0.5  # Should be close to 1

if mean_ok and std_ok:
    print("\n✅ Data standardization looks CORRECT")
    print(f"  Mean is close to 0: {current_mean:.4f}")
    print(f"  Std is close to 1: {current_std:.4f}")
    print("\n  If accuracy is still low, the problem might be:")
    print("    - Different patient population (distribution shift)")
    print("    - Model not generalizing well to new data")
    print("    - Acquisition protocol differences")
else:
    print("\n❌ Data standardization MISMATCH detected!")
    print(f"  Current mean: {current_mean:.4f} (expected ≈ 0.0)")
    print(f"  Current std:  {current_std:.4f} (expected ≈ 1.0)")
    print("\n  This is likely causing the low accuracy!")
    
    print("\n🔧 SOLUTION: Re-standardize the data")
    print("\n  Option 1 - Patient-wise z-score (recommended):")
    print("""
# Re-standardize using patient-wise z-score
features_raw = patient_data['features']
patient_mean = features_raw.mean()
patient_std = features_raw.std()

print(f"Patient mean before: {patient_mean:.6f}")
print(f"Patient std before: {patient_std:.6f}")

features_standardized = (features_raw - patient_mean) / patient_std
patient_data['features'] = features_standardized

print(f"\\nAfter standardization:")
print(f"  Mean: {features_standardized.mean():.6f}")
print(f"  Std: {features_standardized.std():.6f}")

# Then re-run the prediction cells (Cell 7 and Cell 8)
    """)
    
    print("\n  Option 2 - Use training scalers (if available in checkpoint):")
    print("""
# If checkpoint has scaler info, use it
if 'scalers_info' in kan_checkpoint:
    scaler = kan_checkpoint['scalers_info']
    # Apply the same transformation used during training
    # (exact code depends on scaler format)
    """)

print("\n" + "="*80)

In [ ]:
print("\n" + "="*80)
print("🚀 开始预测 - DeepMLP模型")
print("="*80)

# 1. 加载DeepMLP模型
deepmlp_model, deepmlp_checkpoint = load_trained_model(
    model_path=MODELS['DeepMLP']['model_file'],
    model_type=MODELS['DeepMLP']['model_type'],
    device=device
)

# 2. 执行预测
deepmlp_predictions = predict_patient(
    model=deepmlp_model,
    data_info=patient_data,
    device=device,
    batch_size=PREDICTION_CONFIG['batch_size']
)

# 3. 还原为3D volume
deepmlp_volume_3d = predictions_to_3d_volume(
    predictions=deepmlp_predictions,
    data_info=patient_data,
    include_background=PREDICTION_CONFIG['include_background']
)

# 4. 保存结果
deepmlp_nifti_path, deepmlp_json_path = save_prediction_results(
    volume_3d=deepmlp_volume_3d,
    data_info=patient_data,
    output_dir=MODELS['DeepMLP']['output_dir'],
    model_name='deep_mlp',
    include_background=PREDICTION_CONFIG['include_background']
)

# 5. 清理内存 (保留 deepmlp_predictions 供下一个 cell 使用)
del deepmlp_model, deepmlp_volume_3d
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("\n" + "="*80)
print("✅ DeepMLP模型预测完成！")
print(f"💡 deepmlp_predictions 已保留在内存中供 error map 可视化使用")
print("="*80)

In [ ]:
# ==============================================================================
# ERROR MAP VISUALIZATION - INDEPENDENT CELL
# This cell can run independently without previous cells
# ==============================================================================

import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.colors import ListedColormap
import warnings
warnings.filterwarnings('ignore')

# ==============================================================================
# CONFIGURATION - MODIFY THESE PATHS AS NEEDED
# ==============================================================================

# Choose which model's prediction to visualize ('kan' or 'deep_mlp')
MODEL_NAME = 'kan'  # Change to 'deep_mlp' for DeepMLP predictions

# Paths configuration
SERVER_BASE = '/home/jovyan/gpu_space/workspace_jiayi/KAN-git/KAN-Brain-Single-Voxel-Segmentaion/D_proj_analysis/banlanced_sample'

if MODEL_NAME == 'kan':
    PREDICTION_DIR = Path(f'{SERVER_BASE}/refactored_training/training_runs/kan_bg_excl_20251103_162620/results')
elif MODEL_NAME == 'deep_mlp':
    PREDICTION_DIR = Path(f'{SERVER_BASE}/refactored_training/training_runs/deep_mlp_bg_excl_20251103_162620/results')
else:
    raise ValueError(f"Unknown model name: {MODEL_NAME}")

# Ground truth labels path
GT_LABELS_PATH = Path('/home/jovyan/gpu_space/workspace_jiayi/new_datasets/NEW_DATASET_ANALYSIS/FOR_016_20250204_reproducibility/evaluated/resampled_synthseg_t1_mp2rage_alex_labels.nii.gz')

# Visualization parameters
N_SLICES_PER_VIEW = 5  # Number of slices to show per view
FIGSIZE = (20, 12)

print("="*80)
print(f"ERROR MAP VISUALIZATION - {MODEL_NAME.upper()} MODEL")
print("="*80)

# ==============================================================================
# STEP 1: Load Latest Prediction
# ==============================================================================

print("\n📂 Loading prediction files...")

# Find all prediction files for this model
prediction_files = sorted(PREDICTION_DIR.glob(f'prediction_3d_{MODEL_NAME}_*.nii.gz'))

if len(prediction_files) == 0:
    raise FileNotFoundError(f"No prediction files found in {PREDICTION_DIR}")

# Use the latest prediction file
latest_prediction = prediction_files[-1]
print(f"  Found {len(prediction_files)} prediction file(s)")
print(f"  Using latest: {latest_prediction.name}")

# Load prediction softmax (4D: X, Y, Z, n_classes)
pred_img = nib.load(latest_prediction)
pred_softmax = pred_img.get_fdata().astype(np.float32)
print(f"  Prediction shape: {pred_softmax.shape}")

# Get predicted labels (argmax across class dimension)
pred_labels = np.argmax(pred_softmax, axis=-1).astype(np.int32)
print(f"  Predicted labels shape: {pred_labels.shape}")
print(f"  Predicted label range: [{pred_labels.min()}, {pred_labels.max()}]")

# ==============================================================================
# STEP 2: Load Ground Truth Labels
# ==============================================================================

print("\n📂 Loading ground truth labels...")

# Load ground truth labels (3D)
gt_img = nib.load(GT_LABELS_PATH)
gt_labels_raw = gt_img.get_fdata().astype(np.int32)
print(f"  Ground truth shape: {gt_labels_raw.shape}")
print(f"  Ground truth label range (raw): [{gt_labels_raw.min()}, {gt_labels_raw.max()}]")

# Verify shape match
assert pred_labels.shape == gt_labels_raw.shape, \
    f"Shape mismatch: prediction {pred_labels.shape} vs ground truth {gt_labels_raw.shape}"

# ==============================================================================
# STEP 3: Map FreeSurfer Labels to Continuous Indices (0-51)
# ==============================================================================

print("\n🔄 Mapping FreeSurfer labels to continuous indices...")

# FreeSurfer standard labels (52 classes: 0-51)
STANDARD_LABELS = [
    0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 17,
    29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43,
    44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58,
    59, 60, 61, 62, 103
]

# Create forward mapping: FreeSurfer label -> continuous index
forward_mapping = {original: continuous for continuous, original in enumerate(STANDARD_LABELS)}

# Map ground truth labels
gt_labels = np.zeros_like(gt_labels_raw)
unique_gt = np.unique(gt_labels_raw)

for orig_label in unique_gt:
    if orig_label in forward_mapping:
        mask = gt_labels_raw == orig_label
        gt_labels[mask] = forward_mapping[orig_label]

print(f"  Mapped {len(unique_gt)} unique labels")
print(f"  Ground truth label range (mapped): [{gt_labels.min()}, {gt_labels.max()}]")

# ==============================================================================
# STEP 4: Create Error Map
# ==============================================================================

print("\n🎨 Creating error map...")

# Create error map
# 0 = background (gray)
# 1 = correct prediction (green)
# 2 = incorrect prediction (red)

# Initialize as background
error_map = np.zeros_like(pred_labels, dtype=np.uint8)

# Non-background mask (exclude label 0)
non_bg_mask = gt_labels > 0

# Mark correct predictions (green)
correct_mask = (pred_labels == gt_labels) & non_bg_mask
error_map[correct_mask] = 1

# Mark incorrect predictions (red)
incorrect_mask = (pred_labels != gt_labels) & non_bg_mask
error_map[incorrect_mask] = 2

# Calculate accuracy (excluding background)
n_correct = correct_mask.sum()
n_total = non_bg_mask.sum()
gross_accuracy = n_correct / n_total if n_total > 0 else 0.0

print(f"  Total voxels (non-background): {n_total:,}")
print(f"  Correct predictions: {n_correct:,}")
print(f"  Incorrect predictions: {(n_total - n_correct):,}")
print(f"  ✅ Gross Accuracy: {gross_accuracy:.4f} ({gross_accuracy*100:.2f}%)")

# Check if accuracy is suspiciously low
if gross_accuracy < 0.3:
    print(f"\n  ⚠️  WARNING: Accuracy is very low ({gross_accuracy*100:.2f}%)")
    print(f"  This might indicate a data standardization mismatch!")
    print(f"  Training data uses patient-wise standardization.")
    print(f"  Please verify your new data is standardized the same way.")

# ==============================================================================
# STEP 5: Select Slices to Visualize
# ==============================================================================

print(f"\n🖼️  Selecting {N_SLICES_PER_VIEW} slices per view...")

X, Y, Z = pred_labels.shape

# Select evenly spaced slices for each view (avoid edges)
# Axial slices (Z direction)
z_slices = np.linspace(Z//4, 3*Z//4, N_SLICES_PER_VIEW, dtype=int)
z_slices = np.clip(z_slices, 0, Z-1)  # Ensure within bounds

# Coronal slices (Y direction)
y_slices = np.linspace(Y//4, 3*Y//4, N_SLICES_PER_VIEW, dtype=int)
y_slices = np.clip(y_slices, 0, Y-1)

# Sagittal slices (X direction)
x_slices = np.linspace(X//4, 3*X//4, N_SLICES_PER_VIEW, dtype=int)
x_slices = np.clip(x_slices, 0, X-1)

print(f"  Axial (Z) slices: {z_slices.tolist()}")
print(f"  Coronal (Y) slices: {y_slices.tolist()}")
print(f"  Sagittal (X) slices: {x_slices.tolist()}")

# ==============================================================================
# STEP 6: Visualize Error Maps
# ==============================================================================

print("\n📊 Generating visualization...")

# Define colormap: gray (background), green (correct), red (incorrect)
colors = ['#2C2C2C', '#00FF00', '#FF0000']  # dark gray, green, red
cmap = ListedColormap(colors)

# Create figure with 3 rows (one per view)
fig, axes = plt.subplots(3, N_SLICES_PER_VIEW, figsize=FIGSIZE)
fig.suptitle(f'{MODEL_NAME.upper()} Model - Error Map Visualization\n'
             f'Gross Accuracy: {gross_accuracy:.4f} ({gross_accuracy*100:.2f}%)\n'
             f'Green = Correct | Red = Incorrect | Gray = Background',
             fontsize=16, fontweight='bold', y=0.98)

# Plot axial slices (row 0)
for i, z_idx in enumerate(z_slices):
    ax = axes[0, i]
    slice_data = error_map[:, :, z_idx].T  # Transpose for correct orientation
    
    # Calculate slice accuracy (FIXED: removed incorrect transpose)
    slice_mask = gt_labels[:, :, z_idx] > 0
    if slice_mask.sum() > 0:
        slice_correct = (error_map[:, :, z_idx][slice_mask] == 1).sum()
        slice_acc = slice_correct / slice_mask.sum()
    else:
        slice_acc = 0.0
    
    ax.imshow(slice_data, cmap=cmap, vmin=0, vmax=2, interpolation='nearest')
    ax.set_title(f'Axial Z={z_idx}\nAcc: {slice_acc:.3f}', fontsize=10)
    ax.axis('off')

# Plot coronal slices (row 1)
for i, y_idx in enumerate(y_slices):
    ax = axes[1, i]
    slice_data = error_map[:, y_idx, :].T
    
    # Calculate slice accuracy (FIXED)
    slice_mask = gt_labels[:, y_idx, :] > 0
    if slice_mask.sum() > 0:
        slice_correct = (error_map[:, y_idx, :][slice_mask] == 1).sum()
        slice_acc = slice_correct / slice_mask.sum()
    else:
        slice_acc = 0.0
    
    ax.imshow(slice_data, cmap=cmap, vmin=0, vmax=2, interpolation='nearest')
    ax.set_title(f'Coronal Y={y_idx}\nAcc: {slice_acc:.3f}', fontsize=10)
    ax.axis('off')

# Plot sagittal slices (row 2)
for i, x_idx in enumerate(x_slices):
    ax = axes[2, i]
    slice_data = error_map[x_idx, :, :].T
    
    # Calculate slice accuracy (FIXED)
    slice_mask = gt_labels[x_idx, :, :] > 0
    if slice_mask.sum() > 0:
        slice_correct = (error_map[x_idx, :, :][slice_mask] == 1).sum()
        slice_acc = slice_correct / slice_mask.sum()
    else:
        slice_acc = 0.0
    
    ax.imshow(slice_data, cmap=cmap, vmin=0, vmax=2, interpolation='nearest')
    ax.set_title(f'Sagittal X={x_idx}\nAcc: {slice_acc:.3f}', fontsize=10)
    ax.axis('off')

plt.tight_layout()
plt.show()

print("\n" + "="*80)
print("✅ Visualization complete!")
print("="*80)
print(f"\n📊 Summary:")
print(f"  Model: {MODEL_NAME.upper()}")
print(f"  Prediction file: {latest_prediction.name}")
print(f"  Volume shape: {pred_labels.shape}")
print(f"  Total non-background voxels: {n_total:,}")
print(f"  Correct predictions: {n_correct:,} ({n_correct/n_total*100:.2f}%)")
print(f"  Incorrect predictions: {n_total-n_correct:,} ({(n_total-n_correct)/n_total*100:.2f}%)")
print(f"  Overall Accuracy: {gross_accuracy:.4f}")

if gross_accuracy < 0.3:
    print("\n" + "="*80)
    print("⚠️  LOW ACCURACY WARNING")
    print("="*80)
    print("The prediction accuracy is very low, which suggests a possible issue:")
    print("")
    print("LIKELY CAUSE: Data standardization mismatch")
    print("  • Training uses patient-wise z-score: (x - patient_mean) / patient_std")
    print("  • Your new data might use different standardization")
    print("")
    print("SOLUTIONS:")
    print("  1. Check how your new data was standardized")
    print("  2. Re-standardize using the same method as training")
    print("  3. Or use the scaler stored in the .pth checkpoint")
    print("")
    print("See the data standardization section in the prediction notebook.")
    print("="*80)

print("="*80)

## Cell 9: 结果总结

In [ ]:
print("\n" + "="*80)
print("📊 预测结果总结")
print("="*80)

print(f"\n🎯 预测对象: {NEW_PATIENT_DATA['subject_id']}")
print(f"📅 预测时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

print(f"\n📁 生成的文件:")
print("\n1️⃣ KAN模型:")
print(f"   NIfTI: {kan_nifti_path}")
print(f"   大小: {kan_nifti_path.stat().st_size / (1024**2):.2f} MB")
print(f"   JSON: {kan_json_path}")

print(f"\n2️⃣ DeepMLP模型:")
print(f"   NIfTI: {deepmlp_nifti_path}")
print(f"   大小: {deepmlp_nifti_path.stat().st_size / (1024**2):.2f} MB")
print(f"   JSON: {deepmlp_json_path}")

print(f"\n📍 输出目录:")
print(f"   KAN: {MODELS['KAN']['output_dir']}")
print(f"   DeepMLP: {MODELS['DeepMLP']['output_dir']}")

print("\n" + "="*80)
print("✅ 所有预测完成！")
print("="*80)

# 显示如何查看结果
print("\n💡 查看结果:")
print("  在服务器上运行:")
print(f"  ls -lh {MODELS['KAN']['output_dir']}/*.nii.gz | tail -2")
print(f"  ls -lh {MODELS['DeepMLP']['output_dir']}/*.nii.gz | tail -2")

In [ ]:
# ==============================================================================
# ERROR MAP VISUALIZATION - INDEPENDENT CELL
# This cell can run independently without previous cells
# ==============================================================================

import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.colors import ListedColormap
import warnings
warnings.filterwarnings('ignore')

# ==============================================================================
# CONFIGURATION - MODIFY THESE PATHS AS NEEDED
# ==============================================================================

# Choose which model's prediction to visualize ('kan' or 'deep_mlp')
MODEL_NAME = 'kan'  # Change to 'deep_mlp' for DeepMLP predictions

# Paths configuration
SERVER_BASE = '/home/jovyan/gpu_space/workspace_jiayi/KAN-git/KAN-Brain-Single-Voxel-Segmentaion/D_proj_analysis/banlanced_sample'

if MODEL_NAME == 'kan':
    PREDICTION_DIR = Path(f'{SERVER_BASE}/refactored_training/training_runs/kan_bg_excl_20251103_162620/results')
elif MODEL_NAME == 'deep_mlp':
    PREDICTION_DIR = Path(f'{SERVER_BASE}/refactored_training/training_runs/deep_mlp_bg_excl_20251103_162620/results')
else:
    raise ValueError(f"Unknown model name: {MODEL_NAME}")

# Ground truth labels path
GT_LABELS_PATH = Path('/home/jovyan/gpu_space/workspace_jiayi/new_datasets/NEW_DATASET_ANALYSIS/FOR_016_20250204_reproducibility/evaluated/resampled_synthseg_t1_mp2rage_alex_labels.nii.gz')

# Visualization parameters
N_SLICES_PER_VIEW = 5  # Number of slices to show per view
FIGSIZE = (20, 12)

print("="*80)
print(f"ERROR MAP VISUALIZATION - {MODEL_NAME.upper()} MODEL")
print("="*80)

# ==============================================================================
# STEP 1: Load Latest Prediction
# ==============================================================================

print("\n📂 Loading prediction files...")

# Find all prediction files for this model
prediction_files = sorted(PREDICTION_DIR.glob(f'prediction_3d_{MODEL_NAME}_*.nii.gz'))

if len(prediction_files) == 0:
    raise FileNotFoundError(f"No prediction files found in {PREDICTION_DIR}")

# Use the latest prediction file
latest_prediction = prediction_files[-1]
print(f"  Found {len(prediction_files)} prediction file(s)")
print(f"  Using latest: {latest_prediction.name}")

# Load prediction softmax (4D: X, Y, Z, n_classes)
pred_img = nib.load(latest_prediction)
pred_softmax = pred_img.get_fdata().astype(np.float32)
print(f"  Prediction shape: {pred_softmax.shape}")

# Get predicted labels (argmax across class dimension)
pred_labels = np.argmax(pred_softmax, axis=-1).astype(np.int32)
print(f"  Predicted labels shape: {pred_labels.shape}")
print(f"  Predicted label range: [{pred_labels.min()}, {pred_labels.max()}]")

# ==============================================================================
# STEP 2: Load Ground Truth Labels
# ==============================================================================

print("\n📂 Loading ground truth labels...")

# Load ground truth labels (3D)
gt_img = nib.load(GT_LABELS_PATH)
gt_labels_raw = gt_img.get_fdata().astype(np.int32)
print(f"  Ground truth shape: {gt_labels_raw.shape}")
print(f"  Ground truth label range (raw): [{gt_labels_raw.min()}, {gt_labels_raw.max()}]")

# Verify shape match
assert pred_labels.shape == gt_labels_raw.shape, \
    f"Shape mismatch: prediction {pred_labels.shape} vs ground truth {gt_labels_raw.shape}"

# ==============================================================================
# STEP 3: Map FreeSurfer Labels to Continuous Indices (0-51)
# ==============================================================================

print("\n🔄 Mapping FreeSurfer labels to continuous indices...")

# FreeSurfer standard labels (52 classes: 0-51)
STANDARD_LABELS = [
    0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 17,
    29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43,
    44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58,
    59, 60, 61, 62, 103
]

# Create forward mapping: FreeSurfer label -> continuous index
forward_mapping = {original: continuous for continuous, original in enumerate(STANDARD_LABELS)}

# Map ground truth labels
gt_labels = np.zeros_like(gt_labels_raw)
unique_gt = np.unique(gt_labels_raw)

for orig_label in unique_gt:
    if orig_label in forward_mapping:
        mask = gt_labels_raw == orig_label
        gt_labels[mask] = forward_mapping[orig_label]

print(f"  Mapped {len(unique_gt)} unique labels")
print(f"  Ground truth label range (mapped): [{gt_labels.min()}, {gt_labels.max()}]")

# ==============================================================================
# STEP 4: Create Error Map
# ==============================================================================

print("\n🎨 Creating error map...")

# Create error map
# 0 = background (gray)
# 1 = correct prediction (green)
# 2 = incorrect prediction (red)

# Initialize as background
error_map = np.zeros_like(pred_labels, dtype=np.uint8)

# Non-background mask (exclude label 0)
non_bg_mask = gt_labels > 0

# Mark correct predictions (green)
correct_mask = (pred_labels == gt_labels) & non_bg_mask
error_map[correct_mask] = 1

# Mark incorrect predictions (red)
incorrect_mask = (pred_labels != gt_labels) & non_bg_mask
error_map[incorrect_mask] = 2

# Calculate accuracy (excluding background)
n_correct = correct_mask.sum()
n_total = non_bg_mask.sum()
gross_accuracy = n_correct / n_total if n_total > 0 else 0.0

print(f"  Total voxels (non-background): {n_total:,}")
print(f"  Correct predictions: {n_correct:,}")
print(f"  Incorrect predictions: {(n_total - n_correct):,}")
print(f"  ✅ Gross Accuracy: {gross_accuracy:.4f} ({gross_accuracy*100:.2f}%)")

# ==============================================================================
# STEP 5: Select Slices to Visualize
# ==============================================================================

print(f"\n🖼️ Selecting {N_SLICES_PER_VIEW} slices per view...")

X, Y, Z = pred_labels.shape

# Select evenly spaced slices for each view
# Axial slices (Z direction)
z_slices = np.linspace(Z//4, 3*Z//4, N_SLICES_PER_VIEW, dtype=int)

# Coronal slices (Y direction)
y_slices = np.linspace(Y//4, 3*Y//4, N_SLICES_PER_VIEW, dtype=int)

# Sagittal slices (X direction)
x_slices = np.linspace(X//4, 3*X//4, N_SLICES_PER_VIEW, dtype=int)

print(f"  Axial (Z) slices: {z_slices.tolist()}")
print(f"  Coronal (Y) slices: {y_slices.tolist()}")
print(f"  Sagittal (X) slices: {x_slices.tolist()}")

# ==============================================================================
# STEP 6: Visualize Error Maps
# ==============================================================================

print("\n📊 Generating visualization...")

# Define colormap: gray (background), green (correct), red (incorrect)
colors = ['#2C2C2C', '#00FF00', '#FF0000']  # dark gray, green, red
cmap = ListedColormap(colors)

# Create figure with 3 rows (one per view)
fig, axes = plt.subplots(3, N_SLICES_PER_VIEW, figsize=FIGSIZE)
fig.suptitle(f'{MODEL_NAME.upper()} Model - Error Map Visualization\n'
             f'Gross Accuracy: {gross_accuracy:.4f} ({gross_accuracy*100:.2f}%)\n'
             f'Green = Correct | Red = Incorrect | Gray = Background',
             fontsize=16, fontweight='bold', y=0.98)

# Plot axial slices (row 0)
for i, z_idx in enumerate(z_slices):
    ax = axes[0, i]
    slice_data = error_map[:, :, z_idx].T  # Transpose for correct orientation
    
    # Calculate slice accuracy
    slice_mask = gt_labels[:, :, z_idx] > 0
    if slice_mask.sum() > 0:
        slice_acc = (error_map[:, :, z_idx][slice_mask.T] == 1).sum() / slice_mask.sum()
    else:
        slice_acc = 0.0
    
    ax.imshow(slice_data, cmap=cmap, vmin=0, vmax=2, interpolation='nearest')
    ax.set_title(f'Axial Z={z_idx}\nAcc: {slice_acc:.3f}', fontsize=10)
    ax.axis('off')

# Plot coronal slices (row 1)
for i, y_idx in enumerate(y_slices):
    ax = axes[1, i]
    slice_data = error_map[:, y_idx, :].T
    
    # Calculate slice accuracy
    slice_mask = gt_labels[:, y_idx, :] > 0
    if slice_mask.sum() > 0:
        slice_acc = (error_map[:, y_idx, :][slice_mask.T] == 1).sum() / slice_mask.sum()
    else:
        slice_acc = 0.0
    
    ax.imshow(slice_data, cmap=cmap, vmin=0, vmax=2, interpolation='nearest')
    ax.set_title(f'Coronal Y={y_idx}\nAcc: {slice_acc:.3f}', fontsize=10)
    ax.axis('off')

# Plot sagittal slices (row 2)
for i, x_idx in enumerate(x_slices):
    ax = axes[2, i]
    slice_data = error_map[x_idx, :, :].T
    
    # Calculate slice accuracy
    slice_mask = gt_labels[x_idx, :, :] > 0
    if slice_mask.sum() > 0:
        slice_acc = (error_map[x_idx, :, :][slice_mask.T] == 1).sum() / slice_mask.sum()
    else:
        slice_acc = 0.0
    
    ax.imshow(slice_data, cmap=cmap, vmin=0, vmax=2, interpolation='nearest')
    ax.set_title(f'Sagittal X={x_idx}\nAcc: {slice_acc:.3f}', fontsize=10)
    ax.axis('off')

plt.tight_layout()
plt.show()

print("\n" + "="*80)
print("✅ Visualization complete!")
print("="*80)
print(f"\n📊 Summary:")
print(f"  Model: {MODEL_NAME.upper()}")
print(f"  Prediction file: {latest_prediction.name}")
print(f"  Volume shape: {pred_labels.shape}")
print(f"  Total non-background voxels: {n_total:,}")
print(f"  Correct predictions: {n_correct:,} ({n_correct/n_total*100:.2f}%)")
print(f"  Incorrect predictions: {n_total-n_correct:,} ({(n_total-n_correct)/n_total*100:.2f}%)")
print(f"  Overall Accuracy: {gross_accuracy:.4f}")
print("="*80)

In [ ]:
# ==============================================================================
# ERROR MAP VISUALIZATION - INDEPENDENT CELL
# This cell can run independently without previous cells
# ==============================================================================

import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.colors import ListedColormap
import warnings
warnings.filterwarnings('ignore')

# ==============================================================================
# CONFIGURATION - MODIFY THESE PATHS AS NEEDED
# ==============================================================================

# Choose which model's prediction to visualize ('kan' or 'deep_mlp')
MODEL_NAME = 'kan'  # Change to 'deep_mlp' for DeepMLP predictions

# Paths configuration
SERVER_BASE = '/home/jovyan/gpu_space/workspace_jiayi/KAN-git/KAN-Brain-Single-Voxel-Segmentaion/D_proj_analysis/banlanced_sample'

if MODEL_NAME == 'kan':
    PREDICTION_DIR = Path(f'{SERVER_BASE}/refactored_training/training_runs/kan_bg_excl_20251103_162620/results')
elif MODEL_NAME == 'deep_mlp':
    PREDICTION_DIR = Path(f'{SERVER_BASE}/refactored_training/training_runs/deep_mlp_bg_excl_20251103_162620/results')
else:
    raise ValueError(f"Unknown model name: {MODEL_NAME}")

# Ground truth labels path
GT_LABELS_PATH = Path('/home/jovyan/gpu_space/workspace_jiayi/new_datasets/NEW_DATASET_ANALYSIS/FOR_016_20250204_reproducibility/evaluated/resampled_synthseg_t1_mp2rage_alex_labels.nii.gz')

# Visualization parameters
N_SLICES_PER_VIEW = 5  # Number of slices to show per view
FIGSIZE = (20, 12)

print("="*80)
print(f"ERROR MAP VISUALIZATION - {MODEL_NAME.upper()} MODEL")
print("="*80)

# ==============================================================================
# STEP 1: Load Latest Prediction
# ==============================================================================

print("\n📂 Loading prediction files...")

# Find all prediction files for this model
prediction_files = sorted(PREDICTION_DIR.glob(f'prediction_3d_{MODEL_NAME}_*.nii.gz'))

if len(prediction_files) == 0:
    raise FileNotFoundError(f"No prediction files found in {PREDICTION_DIR}")

# Use the latest prediction file
latest_prediction = prediction_files[-1]
print(f"  Found {len(prediction_files)} prediction file(s)")
print(f"  Using latest: {latest_prediction.name}")

# Load prediction softmax (4D: X, Y, Z, n_classes)
pred_img = nib.load(latest_prediction)
pred_softmax = pred_img.get_fdata().astype(np.float32)
print(f"  Prediction shape: {pred_softmax.shape}")

# Get predicted labels (argmax across class dimension)
pred_labels = np.argmax(pred_softmax, axis=-1).astype(np.int32)
print(f"  Predicted labels shape: {pred_labels.shape}")
print(f"  Predicted label range: [{pred_labels.min()}, {pred_labels.max()}]")

# ==============================================================================
# STEP 2: Load Ground Truth Labels
# ==============================================================================

print("\n📂 Loading ground truth labels...")

# Load ground truth labels (3D)
gt_img = nib.load(GT_LABELS_PATH)
gt_labels_raw = gt_img.get_fdata().astype(np.int32)
print(f"  Ground truth shape: {gt_labels_raw.shape}")
print(f"  Ground truth label range (raw): [{gt_labels_raw.min()}, {gt_labels_raw.max()}]")

# Verify shape match
assert pred_labels.shape == gt_labels_raw.shape, \
    f"Shape mismatch: prediction {pred_labels.shape} vs ground truth {gt_labels_raw.shape}"

# ==============================================================================
# STEP 3: Map FreeSurfer Labels to Continuous Indices (0-51)
# ==============================================================================

print("\n🔄 Mapping FreeSurfer labels to continuous indices...")

# FreeSurfer standard labels (52 classes: 0-51)
STANDARD_LABELS = [
    0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 17,
    29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43,
    44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58,
    59, 60, 61, 62, 103
]

# Create forward mapping: FreeSurfer label -> continuous index
forward_mapping = {original: continuous for continuous, original in enumerate(STANDARD_LABELS)}

# Map ground truth labels
gt_labels = np.zeros_like(gt_labels_raw)
unique_gt = np.unique(gt_labels_raw)

for orig_label in unique_gt:
    if orig_label in forward_mapping:
        mask = gt_labels_raw == orig_label
        gt_labels[mask] = forward_mapping[orig_label]

print(f"  Mapped {len(unique_gt)} unique labels")
print(f"  Ground truth label range (mapped): [{gt_labels.min()}, {gt_labels.max()}]")

# ==============================================================================
# STEP 4: Create Error Map
# ==============================================================================

print("\n🎨 Creating error map...")

# Create error map
# 0 = background (gray)
# 1 = correct prediction (green)
# 2 = incorrect prediction (red)

# Initialize as background
error_map = np.zeros_like(pred_labels, dtype=np.uint8)

# Non-background mask (exclude label 0)
non_bg_mask = gt_labels > 0

# Mark correct predictions (green)
correct_mask = (pred_labels == gt_labels) & non_bg_mask
error_map[correct_mask] = 1

# Mark incorrect predictions (red)
incorrect_mask = (pred_labels != gt_labels) & non_bg_mask
error_map[incorrect_mask] = 2

# Calculate accuracy (excluding background)
n_correct = correct_mask.sum()
n_total = non_bg_mask.sum()
gross_accuracy = n_correct / n_total if n_total > 0 else 0.0

print(f"  Total voxels (non-background): {n_total:,}")
print(f"  Correct predictions: {n_correct:,}")
print(f"  Incorrect predictions: {(n_total - n_correct):,}")
print(f"  ✅ Gross Accuracy: {gross_accuracy:.4f} ({gross_accuracy*100:.2f}%)")

# Check if accuracy is suspiciously low
if gross_accuracy < 0.3:
    print(f"\n  ⚠️  WARNING: Accuracy is very low ({gross_accuracy*100:.2f}%)")
    print(f"  This might indicate a data standardization mismatch!")
    print(f"  Training data uses patient-wise standardization.")
    print(f"  Please verify your new data is standardized the same way.")

# ==============================================================================
# STEP 5: Select Slices to Visualize
# ==============================================================================

print(f"\n🖼️  Selecting {N_SLICES_PER_VIEW} slices per view...")

X, Y, Z = pred_labels.shape

# Select evenly spaced slices for each view (avoid edges)
# Axial slices (Z direction)
z_slices = np.linspace(Z//4, 3*Z//4, N_SLICES_PER_VIEW, dtype=int)
z_slices = np.clip(z_slices, 0, Z-1)  # Ensure within bounds

# Coronal slices (Y direction)
y_slices = np.linspace(Y//4, 3*Y//4, N_SLICES_PER_VIEW, dtype=int)
y_slices = np.clip(y_slices, 0, Y-1)

# Sagittal slices (X direction)
x_slices = np.linspace(X//4, 3*X//4, N_SLICES_PER_VIEW, dtype=int)
x_slices = np.clip(x_slices, 0, X-1)

print(f"  Axial (Z) slices: {z_slices.tolist()}")
print(f"  Coronal (Y) slices: {y_slices.tolist()}")
print(f"  Sagittal (X) slices: {x_slices.tolist()}")

# ==============================================================================
# STEP 6: Visualize Error Maps
# ==============================================================================

print("\n📊 Generating visualization...")

# Define colormap: gray (background), green (correct), red (incorrect)
colors = ['#2C2C2C', '#00FF00', '#FF0000']  # dark gray, green, red
cmap = ListedColormap(colors)

# Create figure with 3 rows (one per view)
fig, axes = plt.subplots(3, N_SLICES_PER_VIEW, figsize=FIGSIZE)
fig.suptitle(f'{MODEL_NAME.upper()} Model - Error Map Visualization\n'
             f'Gross Accuracy: {gross_accuracy:.4f} ({gross_accuracy*100:.2f}%)\n'
             f'Green = Correct | Red = Incorrect | Gray = Background',
             fontsize=16, fontweight='bold', y=0.98)

# Plot axial slices (row 0)
for i, z_idx in enumerate(z_slices):
    ax = axes[0, i]
    slice_data = error_map[:, :, z_idx].T  # Transpose for correct orientation
    
    # Calculate slice accuracy (FIXED: removed incorrect transpose)
    slice_mask = gt_labels[:, :, z_idx] > 0
    if slice_mask.sum() > 0:
        slice_correct = (error_map[:, :, z_idx][slice_mask] == 1).sum()
        slice_acc = slice_correct / slice_mask.sum()
    else:
        slice_acc = 0.0
    
    ax.imshow(slice_data, cmap=cmap, vmin=0, vmax=2, interpolation='nearest')
    ax.set_title(f'Axial Z={z_idx}\nAcc: {slice_acc:.3f}', fontsize=10)
    ax.axis('off')

# Plot coronal slices (row 1)
for i, y_idx in enumerate(y_slices):
    ax = axes[1, i]
    slice_data = error_map[:, y_idx, :].T
    
    # Calculate slice accuracy (FIXED)
    slice_mask = gt_labels[:, y_idx, :] > 0
    if slice_mask.sum() > 0:
        slice_correct = (error_map[:, y_idx, :][slice_mask] == 1).sum()
        slice_acc = slice_correct / slice_mask.sum()
    else:
        slice_acc = 0.0
    
    ax.imshow(slice_data, cmap=cmap, vmin=0, vmax=2, interpolation='nearest')
    ax.set_title(f'Coronal Y={y_idx}\nAcc: {slice_acc:.3f}', fontsize=10)
    ax.axis('off')

# Plot sagittal slices (row 2)
for i, x_idx in enumerate(x_slices):
    ax = axes[2, i]
    slice_data = error_map[x_idx, :, :].T
    
    # Calculate slice accuracy (FIXED)
    slice_mask = gt_labels[x_idx, :, :] > 0
    if slice_mask.sum() > 0:
        slice_correct = (error_map[x_idx, :, :][slice_mask] == 1).sum()
        slice_acc = slice_correct / slice_mask.sum()
    else:
        slice_acc = 0.0
    
    ax.imshow(slice_data, cmap=cmap, vmin=0, vmax=2, interpolation='nearest')
    ax.set_title(f'Sagittal X={x_idx}\nAcc: {slice_acc:.3f}', fontsize=10)
    ax.axis('off')

plt.tight_layout()
plt.show()

print("\n" + "="*80)
print("✅ Visualization complete!")
print("="*80)
print(f"\n📊 Summary:")
print(f"  Model: {MODEL_NAME.upper()}")
print(f"  Prediction file: {latest_prediction.name}")
print(f"  Volume shape: {pred_labels.shape}")
print(f"  Total non-background voxels: {n_total:,}")
print(f"  Correct predictions: {n_correct:,} ({n_correct/n_total*100:.2f}%)")
print(f"  Incorrect predictions: {n_total-n_correct:,} ({(n_total-n_correct)/n_total*100:.2f}%)")
print(f"  Overall Accuracy: {gross_accuracy:.4f}")

if gross_accuracy < 0.3:
    print("\n" + "="*80)
    print("⚠️  LOW ACCURACY WARNING")
    print("="*80)
    print("The prediction accuracy is very low, which suggests a possible issue:")
    print("")
    print("LIKELY CAUSE: Data standardization mismatch")
    print("  • Training uses patient-wise z-score: (x - patient_mean) / patient_std")
    print("  • Your new data might use different standardization")
    print("")
    print("SOLUTIONS:")
    print("  1. Check how your new data was standardized")
    print("  2. Re-standardize using the same method as training")
    print("  3. Or use the scaler stored in the .pth checkpoint")
    print("")
    print("See the data standardization section in the prediction notebook.")
    print("="*80)

print("="*80)